In [4]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cargar los 120 archivos JSONL
file_paths = glob.glob("../data/raw/usage_events_stream/*.jsonl")

records = []
for path in file_paths:
    file_name = os.path.basename(path)
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                data['_source_file'] = file_name
                records.append(data)

df = pd.DataFrame(records)
df['event_timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Extracción de fecha/hora del nombre del micro-lote para medir late-data
df['_ingest_ts'] = pd.to_datetime(
    df['_source_file'].str.extract(r'(\d{8}_\d{6})')[0],
    format='%Y%m%d_%H%M%S',
    errors='coerce'
)

# 3. Medición del desorden temporal (Diferencia Ingesta vs Evento Real)
df['delay_hours'] = (df['_ingest_ts'] - df['event_timestamp']).dt.total_seconds() / 3600.0

# PRINT DE RESULTADOS CLAVE
print(f"Total de registros analizados: {len(df)}")
print(f"Duplicados de event_id: {df.duplicated('event_id').sum()}")
print(f"\nDistribución de Schema Version:\n{df['schema_version'].value_counts(dropna=False)}")
print(f"\nNulos en V2 (carbon_kg): {df['carbon_kg'].isna().sum()}")
print(f"Nulos en V2 (genai_tokens): {df['genai_tokens'].isna().sum()}")

print("\n--- PERCENTILES DE RETRASO TEMPORAL (LATE DATA) ---")
print(f"Percentil 50 (Mediana): {df['delay_hours'].median():.2f} horas")
print(f"Percentil 95: {df['delay_hours'].quantile(0.95):.2f} horas")
print(f"Percentil 99: {df['delay_hours'].quantile(0.99):.2f} horas")
print(f"Máximo retraso detectado: {df['delay_hours'].max():.2f} horas")

KeyError: 'timestamp'